In [1]:
from ase.db import connect
from ase.io import write
from ase.calculators.orca import ORCA
from ase.vibrations import Vibrations
import numpy as np
import os
from collections import Counter

In [2]:
data_dir = os.path.expanduser("~/Desktop/train_4M")
shards = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".aselmdb")]
print(f"Found {len(shards)} shards")
db = connect(shards[0])

Found 80 shards


In [3]:
def load_sample(row):
    atoms = row.toatoms()
    return {
        "positions": atoms.positions,          # (N, 3)
        "atomic_numbers": atoms.get_atomic_numbers(),
        "energy": row.energy,
        "forces": row.forces,                  # (N, 3)
        "charge": row.data["charge"],
        "spin": row.data["spin"],
        "domain": row.data["data_id"],
        "formula": row.data["composition"],
    }

In [4]:
samples = [load_sample(row) for row in db.select(limit=10)]
domains = Counter(row.data["data_id"] for row in db.select())
print(domains)

Counter({'metal_complexes': 10083, 'elytes': 9939, 'reactivity': 9927, 'biomolecules': 9890, 'trans1x': 2730, 'ani2x': 2648, 'geom_orca6': 2504, 'rgd': 893, 'orbnet_denali': 647, 'spice': 574})


In [10]:
def is_usable(row):
    if row.data["num_atoms"] > 30:
        return False
    return True

In [11]:
row = next(row for row in db.select() if is_usable(row))
atoms = row.toatoms()
print(f"Selected: {row.data['composition']}, {row.data['num_atoms']} atoms, domain: {row.data['data_id']}")


Selected: C4H10N2O1, 17 atoms, domain: biomolecules


In [ ]:
atoms.calc = ORCA(
    label="molecule",
    orcasimpleinput="wB97M-V def2-TZVPD TightSCF",
    orcablocks=f"%pal nprocs 8 end\n%scf MaxIter 300 end",
    charge=row.data["charge"],
    mult=row.data["spin"],  # spin in OMol25 is already multiplicity (2S+1)
)

os.makedirs("vib_output", exist_ok=True)
vib = Vibrations(atoms, name="vib_output/vib")
vib.run()
vib.summary()

In [ ]:
# Need to download ORCA/use SLAC resources I guess to run this stuff... suppposed to be finicky on macs 
T = 300 
kB = 8.617e-5
n_frames = 100
output_dir = "trajectories"
os.makedirs(output_dir, exist_ok=True)

freqs = vib.get_frequencies()

for k in range(6, 3 * row.data["num_atoms"]):
    omega_cm = freqs[k].real

    if omega_cm <= 0:
        continue

    omega_rad_per_s = omega_cm * 2 * np.pi * 3e10  
    A = np.sqrt(kB * T) / omega_rad_per_s    

    mode = vib.get_mode(k) 

    frames = []
    period = 1.0 / (omega_cm * 3e10) 
    for t in np.linspace(0, period, n_frames):
        displaced = atoms.copy()
        displaced.positions += A * mode * np.cos(2 * np.pi * omega_cm * 3e10 * t)
        frames.append(displaced)

    write(os.path.join(output_dir, f"mode_{k:03d}_omega_{omega_cm:.1f}cm.xyz"), frames)
    print(f"Mode {k}: {omega_cm:.1f} cm^-1, amplitude {A*1e10:.3f} Angstrom")